In [0]:
# ============================================================
# Retail Intelligence Brasil
# Notebook.......: 16_bronze_sidra_populacao_sexo_idade
# Camada.........: Bronze
# Fonte..........: IBGE - SIDRA
# Tabela.........: 9514
# Descrição......: População residente por sexo e faixa etária
# Objetivo.......: Extrair dados demográficos do SIDRA,
#                  consolidar em JSON na Landing e persistir
#                  na camada Bronze em Delta Lake.
#
# Autor..........: Cristhina Freire
# Data...........: 2026
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import requests

from pyspark.sql import SparkSession

# ============================================================
# 2. CONFIGURAÇÃO
# ============================================================

CATALOG = "retail_intelligence"
SCHEMA = "bronze"

TABLE_NAME = "ibge_sidra_populacao_sexo_idade_raw"

FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

BASE_URL = "https://apisidra.ibge.gov.br"

SIDRA_TABLE_ID = 9514

TIMEOUT = 60

CURRENT_DATE = datetime.now().strftime("%Y-%m-%d")
CURRENT_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/landing/"
    f"ibge/sidra/demografia/populacao_sexo_idade/{CURRENT_DATE}"
)

FILE_NAME = f"populacao_sexo_idade_{CURRENT_TIME}.json"

FILE_PATH = f"{VOLUME_PATH}/{FILE_NAME}"

# ============================================================
# CAMINHOS
# ============================================================

LANDING_PATH = (
    "/Volumes/retail_intelligence/"
    "bronze/landing/ibge/"
    "sidra/demografia/"
    "populacao_sexo_idade/"
)

BRONZE_PATH = (
    "/Volumes/retail_intelligence/"
    "bronze/ibge/"
    "sidra/populacao_sexo_idade"
)

ARQUIVO_JSON = "populacao_sexo_idade_2022.json"

# ============================================================
# SPARK
# ============================================================

spark = SparkSession.builder.getOrCreate()

# ============================================================
# LOG
# ============================================================

print("=" * 70)
print("SIDRA 9514 - POPULAÇÃO POR SEXO E FAIXA ETÁRIA")
print("=" * 70)

print(f"Tabela SIDRA : {SIDRA_TABLE}")
print(f"Landing      : {LANDING_PATH}")
print(f"Bronze       : {BRONZE_PATH}")

SIDRA 9514 - POPULAÇÃO POR SEXO E FAIXA ETÁRIA
Tabela SIDRA : 9514
Landing      : /Volumes/retail_intelligence/bronze/landing/ibge/sidra/demografia/populacao_sexo_idade/
Bronze       : /Volumes/retail_intelligence/bronze/ibge/sidra/populacao_sexo_idade


In [0]:
# ============================================================
# 3. FAIXAS ETÁRIAS (GRUPOS QUINQUENAIS)
# ============================================================

FAIXAS_ETARIAS = [
    {"codigo": 93070, "descricao": "0 a 4 anos"},
    {"codigo": 93084, "descricao": "5 a 9 anos"},
    {"codigo": 93085, "descricao": "10 a 14 anos"},
    {"codigo": 93086, "descricao": "15 a 19 anos"},
    {"codigo": 93087, "descricao": "20 a 24 anos"},
    {"codigo": 93088, "descricao": "25 a 29 anos"},
    {"codigo": 93089, "descricao": "30 a 34 anos"},
    {"codigo": 93090, "descricao": "35 a 39 anos"},
    {"codigo": 93091, "descricao": "40 a 44 anos"},
    {"codigo": 93092, "descricao": "45 a 49 anos"},
    {"codigo": 93093, "descricao": "50 a 54 anos"},
    {"codigo": 93094, "descricao": "55 a 59 anos"},
    {"codigo": 93095, "descricao": "60 a 64 anos"},
    {"codigo": 93096, "descricao": "65 a 69 anos"},
    {"codigo": 93097, "descricao": "70 a 74 anos"},
    {"codigo": 93098, "descricao": "75 a 79 anos"},
    {"codigo": 49108, "descricao": "80 a 84 anos"},
    {"codigo": 49109, "descricao": "85 a 89 anos"},
    {"codigo": 60040, "descricao": "90 a 94 anos"},
    {"codigo": 60041, "descricao": "95 a 99 anos"},
    {"codigo": 6653, "descricao": "100 anos ou mais"}
]

print(f"Total de faixas etárias: {len(FAIXAS_ETARIAS)}")

Total de faixas etárias: 21


In [0]:
# ============================================================
# 4. FUNÇÃO DE EXTRAÇÃO SIDRA
# ============================================================

def consultar_sidra(codigo_faixa):

    url = (
        f"{BASE_URL}/values/"
        f"t/{SIDRA_TABLE}/"
        f"n6/all/"
        f"c2/all/"
        f"c287/{codigo_faixa}"
    )

    response = requests.get(
        url,
        timeout=TIMEOUT
    )

    if response.status_code != 200:
        raise Exception(
            f"Erro {response.status_code} na consulta da faixa {codigo_faixa}"
        )

    return response.json()

In [0]:
# ============================================================
# 4. FUNÇÃO DE EXTRAÇÃO SIDRA
# ============================================================

def consultar_sidra(
    base_url,
    tabela,
    codigo_faixa,
    timeout=60
):

    url = (
        f"{base_url}/values/"
        f"t/{tabela}/"
        f"n6/all/"
        f"c2/all/"
        f"c287/{codigo_faixa}"
    )

    response = requests.get(
        url,
        timeout=timeout
    )

    if response.status_code != 200:
        raise Exception(
            f"Erro {response.status_code} na consulta da faixa {codigo_faixa}"
        )

    return response.json()

In [0]:
# ============================================================
# 6. EXTRAÇÃO DOS DADOS
# ============================================================

dados_consolidados = []

total_requisicoes = 0

for faixa in FAIXAS_ETARIAS:

    codigo = faixa["codigo"]
    descricao = faixa["descricao"]

    print(f"Extraindo faixa etária: {descricao} ({codigo})...")

    resposta = consultar_sidra(
        BASE_URL,
        SIDRA_TABLE,
        codigo,
        TIMEOUT
    )

    # Remove o cabeçalho da resposta
    registros = resposta[1:]

    dados_consolidados.extend(registros)

    total_requisicoes += 1

    print(f"Registros adicionados: {len(registros):,}")

print("\nExtração finalizada.")

Extraindo faixa etária: 0 a 4 anos (93070)...
Registros adicionados: 16,710
Extraindo faixa etária: 5 a 9 anos (93084)...
Registros adicionados: 16,710
Extraindo faixa etária: 10 a 14 anos (93085)...
Registros adicionados: 16,710
Extraindo faixa etária: 15 a 19 anos (93086)...
Registros adicionados: 16,710
Extraindo faixa etária: 20 a 24 anos (93087)...
Registros adicionados: 16,710
Extraindo faixa etária: 25 a 29 anos (93088)...
Registros adicionados: 16,710
Extraindo faixa etária: 30 a 34 anos (93089)...
Registros adicionados: 16,710
Extraindo faixa etária: 35 a 39 anos (93090)...
Registros adicionados: 16,710
Extraindo faixa etária: 40 a 44 anos (93091)...
Registros adicionados: 16,710
Extraindo faixa etária: 45 a 49 anos (93092)...
Registros adicionados: 16,710
Extraindo faixa etária: 50 a 54 anos (93093)...
Registros adicionados: 16,710
Extraindo faixa etária: 55 a 59 anos (93094)...
Registros adicionados: 16,710
Extraindo faixa etária: 60 a 64 anos (93095)...
Registros adicionado

In [0]:
# ============================================================
# 7. VALIDAÇÃO DA EXTRAÇÃO
# ============================================================

print("=" * 70)

print(f"Requisições realizadas : {total_requisicoes}")
print(f"Registros consolidados : {len(dados_consolidados):,}")

municipios = {r["D1C"] for r in dados_consolidados}
sexos = {r["D2N"] for r in dados_consolidados}
faixas = {r["D3N"] for r in dados_consolidados}

print(f"Municípios distintos   : {len(municipios):,}")
print(f"Sexos encontrados      : {sorted(sexos)}")
print(f"Faixas etárias         : {len(faixas)}")

print("=" * 70)

Requisições realizadas : 21
Registros consolidados : 350,910
Municípios distintos   : 5,570
Sexos encontrados      : ['Homens', 'Mulheres', 'Total']
Faixas etárias         : 21


In [0]:
# ============================================================
# 8. CRIAR DATAFRAME SPARK
# ============================================================

df = spark.createDataFrame(dados_consolidados)

print("=" * 70)
print("DATAFRAME CRIADO")
print("=" * 70)

print(f"Total de registros: {df.count():,}")

display(df.limit(10))

DATAFRAME CRIADO
Total de registros: 350,910


D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N,D5C,D5N,D6C,D6N,MC,MN,NC,NN,V
1100015,Alta Floresta D'Oeste - RO,6794,Total,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,1564
1100015,Alta Floresta D'Oeste - RO,4,Homens,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,796
1100015,Alta Floresta D'Oeste - RO,5,Mulheres,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,768
1100023,Ariquemes - RO,6794,Total,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,6946
1100023,Ariquemes - RO,4,Homens,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,3489
1100023,Ariquemes - RO,5,Mulheres,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,3457
1100031,Cabixi - RO,6794,Total,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,331
1100031,Cabixi - RO,4,Homens,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,175
1100031,Cabixi - RO,5,Mulheres,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,156
1100049,Cacoal - RO,6794,Total,93070,0 a 4 anos,2022,2022,93,População residente,113635,Total,45,Pessoas,6,Município,5792


In [0]:
# ============================================================
# 9. SCHEMA
# ============================================================

df.printSchema()

root
 |-- D1C: string (nullable = true)
 |-- D1N: string (nullable = true)
 |-- D2C: string (nullable = true)
 |-- D2N: string (nullable = true)
 |-- D3C: string (nullable = true)
 |-- D3N: string (nullable = true)
 |-- D4C: string (nullable = true)
 |-- D4N: string (nullable = true)
 |-- D5C: string (nullable = true)
 |-- D5N: string (nullable = true)
 |-- D6C: string (nullable = true)
 |-- D6N: string (nullable = true)
 |-- MC: string (nullable = true)
 |-- MN: string (nullable = true)
 |-- NC: string (nullable = true)
 |-- NN: string (nullable = true)
 |-- V: string (nullable = true)



In [0]:
import json
import os
import requests

from datetime import datetime

from pyspark.sql import SparkSession

In [0]:
from datetime import datetime

CURRENT_DATE = datetime.now().strftime("%Y%m%d")

In [0]:
# ============================================================
# LANDING
# ============================================================

CURRENT_DATE = datetime.now().strftime("%Y%m%d")

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/landing/"
    f"ibge/sidra/demografia/populacao_sexo_idade/{CURRENT_DATE}"
)

ARQUIVO_JSON = "populacao_sexo_idade_2022.json"

# ============================================================
# BRONZE
# ============================================================

BRONZE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/bronze/"
    f"ibge/sidra/populacao_sexo_idade"
)

In [0]:
# ============================================================
# 10. PERSISTÊNCIA DO JSON
# ============================================================

print("Salvando JSON na Landing...")

dbutils.fs.mkdirs(VOLUME_PATH)

dbutils.fs.put(
    FILE_PATH,
    json.dumps(
        dados_consolidados,
        ensure_ascii=False,
        indent=2
    ),
    overwrite=True
)

print(f"Arquivo salvo em:\n{FILE_PATH}")

Salvando JSON na Landing...
Wrote 130274599 bytes.
Arquivo salvo em:
/Volumes/retail_intelligence/bronze/landing/ibge/sidra/demografia/populacao_sexo_idade/2026-07-14/populacao_sexo_idade_20260714_210453.json


In [0]:
# ============================================================
# 11. LEITURA DO JSON
# ============================================================

print("Lendo JSON salvo...")

landing_df = (
    spark.read
         .option("multiline", "true")
         .json(FILE_PATH)
)

print("Leitura concluída.")

Lendo JSON salvo...
Leitura concluída.


In [0]:
# ============================================================
# 12. PERSISTÊNCIA NA BRONZE
# ============================================================

print("Persistindo tabela Delta...")

(
    landing_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FULL_TABLE_NAME)
)

print("Tabela criada com sucesso.")

Persistindo tabela Delta...
Tabela criada com sucesso.
